# 02 — Auxotrophy benchmark reproduction

**Workflow version:** 0.5.5

Reproduce the 147 gene-compound benchmark with strict pair isolation. Each released SBML model is parsed once into a solver-independent COBRA JSON template. Every phenotype is then executed in a fresh Python process that reconstructs a new COBRA model and a new optimization backend from that JSON template. This avoids both repeated libSBML parsing and reuse/copying of solver state.


In [1]:
# Code Cell 1 — Setup imports, protocol constants, and paths
# Requires: nothing
# Provides: protocol constants and repository paths

from pathlib import Path
import hashlib
import json
import platform
import re
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import cobra
from cobra.io import (
    load_json_model,
    read_sbml_model,
    save_json_model,
)

WORKFLOW_VERSION = "0.5.8"
SOLVER = "glpk"
UPTAKE_LOWER_BOUND = -1000.0
VIABILITY_FRACTION = 0.01
ISOLATION_MODE = "fresh_process_json_condition_isolated"
SOLVER_TIMEOUT_SECONDS = 180
PROCESS_TIMEOUT_SECONDS = 240
PRIMARY_FEASIBILITY_TOLERANCE = 1e-7
RETRY_FEASIBILITY_TOLERANCE = 1e-9
RETRY_PROCESS_TIMEOUT_SECONDS = 120
RETRY_POLICY = "process_timeout_only_tight_feasibility"

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = ROOT / "data" / "raw"
RESULTS_DIR = ROOT / "results"
CACHE_DIR = RESULTS_DIR / "model_cache"

RESULTS_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

ORIGINAL_XML = DATA_DIR / "yeast9.0.xml"
CURATED_XML = DATA_DIR / "Yeast9_curated.xml"
DATASET_XLSX = DATA_DIR / "mmc3.xlsx"

ORIGINAL_JSON = CACHE_DIR / "Yeast9.from_sbml.json"
CURATED_JSON = CACHE_DIR / "Yeast9_curated.from_sbml.json"

PROCESS_WORKER = ROOT / "scripts" / "json_pair_worker.py"

ALIASES = {
    "Yeast9": {},
    "Yeast9_curated": {"a_0001": "r_temp1"},
}

if not PROCESS_WORKER.exists():
    raise FileNotFoundError(
        f"Missing process worker: {PROCESS_WORKER}"
    )


In [2]:
# Code Cell 2 — Parse Dataset 2 into stable phenotype records
# Requires: Code Cell 1
# Provides: pairs with 147 benchmark records

REQUIRED_COLUMNS = [
    "Gene Systematic Name", "Chemical", "exchange", "ID",
    "Strain Background", "Reference",
]

def split_plus(value):
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split("+") if part.strip()]

def split_genes(value):
    if pd.isna(value):
        return []
    return [
        part.strip()
        for part in re.split(r"\s+and\s+", str(value).strip(), flags=re.I)
        if part.strip()
    ]

def count_exchange_targets(value):
    if pd.isna(value):
        return 0
    text = str(value).strip()
    return len(re.findall(r"\bexchange\b", text, flags=re.I))

def parse_dataset(path: Path) -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name="all")
    missing = [column for column in REQUIRED_COLUMNS if column not in raw.columns]
    if missing:
        raise ValueError(f"Missing Dataset 2 columns: {missing}")

    records = []
    for idx, row in raw.iterrows():
        excel_row = idx + 2
        genes = split_genes(row["Gene Systematic Name"])
        ids = split_plus(row["ID"])
        chemical = str(row["Chemical"]).strip()
        n_targets = count_exchange_targets(row["exchange"])
        conditional = bool(re.search(r"\badd\b", chemical, flags=re.I))

        if not genes or not ids or n_targets < 1 or len(ids) < n_targets:
            raise ValueError(f"Cannot parse Dataset 2 Excel row {excel_row}.")

        rescue_ids = ids[:n_targets] if conditional else ids
        background_ids = ids[n_targets:] if conditional else []

        records.append({
            "pair_id": f"excel:{excel_row}",
            "excel_row": excel_row,
            "gene_field": str(row["Gene Systematic Name"]).strip(),
            "genes": genes,
            "n_genes": len(genes),
            "chemical": chemical,
            "exchange_field": str(row["exchange"]).strip(),
            "id_field": str(row["ID"]).strip(),
            "rescue_ids": rescue_ids,
            "background_ids": background_ids,
            "conditional_medium": conditional,
            "strain_background": str(row["Strain Background"]).strip(),
            "reference": str(row["Reference"]).strip(),
        })

    pairs = pd.DataFrame(records)

    if len(pairs) != 147:
        raise AssertionError(f"Expected 147 records, found {len(pairs)}.")

    if pairs["pair_id"].duplicated().any():
        raise AssertionError("pair_id values are not unique.")

    return pairs

pairs = parse_dataset(DATASET_XLSX)

display(pairs.head())
print("Records:", len(pairs))
print("Conditional-medium records:", int(pairs["conditional_medium"].sum()))
print("Multi-gene records:", int((pairs["n_genes"] > 1).sum()))


   pair_id  excel_row gene_field      genes  n_genes  chemical       exchange_field id_field rescue_ids background_ids  conditional_medium strain_background                                          reference
0  excel:2          2    YOL058W  [YOL058W]        1  arginine  L-arginine exchange   r_1879   [r_1879]             []               False        Sigma1278b              Crabeel M, et al. (1988) PMID:2897249
1  excel:3          3    YJL071W  [YJL071W]        1  arginine  L-arginine exchange   r_1879   [r_1879]             []               False             S288C  Vandenbol M and Portetelle D (1999) PMID:10509023
2  excel:4          4    YJL088W  [YJL088W]        1  arginine  L-arginine exchange   r_1879   [r_1879]             []               False             Other               Cheng MY, et al. (1987) PMID:3295876
3  excel:5          5    YHR018C  [YHR018C]        1  arginine  L-arginine exchange   r_1879   [r_1879]             []               False             Other      Espino

In [3]:
# Code Cell 3 — Fresh JSON reconstruction and phenotype helpers
# Requires: Code Cells 1–2
# Provides: template validation and pair-isolated worker execution

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)

    return digest.hexdigest()


def get_git_commit(root: Path) -> str:
    completed = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=root,
        capture_output=True,
        text=True,
        check=True,
    )

    return completed.stdout.strip()


def format_duration(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    if hours:
        return f"{hours:02d}h {minutes:02d}m {seconds:02d}s"

    return f"{minutes:02d}m {seconds:02d}s"


def format_growth(value):
    if value is None:
        return "nan"

    value = float(value)

    if not np.isfinite(value):
        return "nan"

    return f"{value:.6g}"


def objective_signature(model):
    return sorted(
        (
            reaction.id,
            float(reaction.objective_coefficient),
        )
        for reaction in model.reactions
        if reaction.objective_coefficient != 0
    )


def prepare_json_template(
    sbml_path,
    json_path,
    model_label,
):
    source_model = read_sbml_model(
        str(sbml_path)
    )
    source_model.solver = SOLVER

    source_growth = source_model.slim_optimize(
        error_value=np.nan
    )

    if str(
        source_model.solver.status
    ).lower() != "optimal":
        raise RuntimeError(
            f"{model_label}: source SBML WT optimization failed."
        )

    source_dimensions = (
        len(source_model.reactions),
        len(source_model.metabolites),
        len(source_model.genes),
    )
    source_objective = objective_signature(
        source_model
    )

    save_json_model(
        source_model,
        json_path,
        sort=False,
    )

    rebuilt_model = load_json_model(
        json_path
    )
    rebuilt_model.solver = SOLVER

    rebuilt_growth = rebuilt_model.slim_optimize(
        error_value=np.nan
    )

    rebuilt_dimensions = (
        len(rebuilt_model.reactions),
        len(rebuilt_model.metabolites),
        len(rebuilt_model.genes),
    )
    rebuilt_objective = objective_signature(
        rebuilt_model
    )

    if rebuilt_dimensions != source_dimensions:
        raise RuntimeError(
            f"{model_label}: JSON reconstruction changed model dimensions."
        )

    if rebuilt_objective != source_objective:
        raise RuntimeError(
            f"{model_label}: JSON reconstruction changed the objective."
        )

    if not np.isclose(
        rebuilt_growth,
        source_growth,
        rtol=0.0,
        atol=1e-10,
    ):
        raise RuntimeError(
            f"{model_label}: JSON reconstruction changed WT growth."
        )

    return {
        "model": model_label,
        "sbml_sha256": sha256_file(
            Path(sbml_path)
        ),
        "json_sha256": sha256_file(
            Path(json_path)
        ),
        "dimensions": {
            "reactions": source_dimensions[0],
            "metabolites": source_dimensions[1],
            "genes": source_dimensions[2],
        },
        "objective": source_objective,
        "source_wt_growth": float(
            source_growth
        ),
        "json_wt_growth": float(
            rebuilt_growth
        ),
    }


def validate_curated_thi6_sentinel(
    model_json,
    threshold,
):
    model = load_json_model(
        model_json
    )
    model.solver = SOLVER

    try:
        model.solver.configuration.timeout = (
            SOLVER_TIMEOUT_SECONDS
        )
    except Exception:
        pass

    gene = model.genes.get_by_id(
        "YPL214C"
    )
    gene.knock_out()

    expected_reactions = [
        "r_0556",
        "r_1036",
        "r_4753",
    ]

    bounds = {
        reaction_id:
        model.reactions.get_by_id(
            reaction_id
        ).bounds
        for reaction_id
        in expected_reactions
    }

    growth = model.slim_optimize(
        error_value=np.nan
    )
    status = str(
        model.solver.status
    ).lower()

    if status != "optimal":
        raise RuntimeError(
            "Curated THI6 JSON sentinel did not solve optimally."
        )

    if not all(
        bound == (0, 0)
        for bound in bounds.values()
    ):
        raise RuntimeError(
            "Curated THI6 JSON sentinel did not close all THI6 reactions."
        )

    if not np.isfinite(growth) or growth >= threshold:
        raise RuntimeError(
            "Curated THI6 JSON sentinel remained viable after knockout."
        )

    return {
        "gene": "YPL214C",
        "status": status,
        "growth": float(growth),
        "bounds": {
            key: list(value)
            for key, value
            in bounds.items()
        },
    }


def parse_worker_json(
    stdout,
    pair_id,
):
    lines = [
        line.strip()
        for line in stdout.splitlines()
        if line.strip()
    ]

    for line in reversed(lines):
        try:
            payload = json.loads(line)
        except json.JSONDecodeError:
            continue

        if (
            isinstance(payload, dict)
            and payload.get("pair_id") == pair_id
        ):
            return payload

    raise RuntimeError(
        f"Worker returned no valid JSON result for {pair_id}."
    )


def run_fresh_json_pair(
    model_json,
    model_label,
    record,
    threshold,
    aliases,
):
    base_command = [
        sys.executable,
        str(PROCESS_WORKER),
        "--model-json",
        str(model_json),
        "--model-label",
        model_label,
        "--solver",
        SOLVER,
        "--solver-timeout",
        str(SOLVER_TIMEOUT_SECONDS),
        "--threshold",
        repr(float(threshold)),
        "--uptake-lower-bound",
        repr(float(UPTAKE_LOWER_BOUND)),
        "--aliases-json",
        json.dumps(aliases),
        "--pair-id",
        record["pair_id"],
    ]

    for gene_id in record["genes"]:
        base_command.extend([
            "--gene",
            gene_id,
        ])

    for reaction_id in record["background_ids"]:
        base_command.extend([
            "--background",
            reaction_id,
        ])

    for reaction_id in record["rescue_ids"]:
        base_command.extend([
            "--rescue",
            reaction_id,
        ])

    def execute_worker(
        feasibility_tolerance,
        process_timeout_seconds,
    ):
        command = base_command + [
            "--feasibility-tolerance",
            repr(float(feasibility_tolerance)),
        ]

        return subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=process_timeout_seconds,
            check=False,
        )

    primary_process_timeout = False
    retry_attempted = False
    retry_status = "not_attempted"
    result_source = "primary"

    try:
        completed = execute_worker(
            PRIMARY_FEASIBILITY_TOLERANCE,
            PROCESS_TIMEOUT_SECONDS,
        )
    except subprocess.TimeoutExpired:
        primary_process_timeout = True
        retry_attempted = True
        result_source = "retry"

        try:
            completed = execute_worker(
                RETRY_FEASIBILITY_TOLERANCE,
                RETRY_PROCESS_TIMEOUT_SECONDS,
            )
        except subprocess.TimeoutExpired:
            retry_status = "process_timeout"
            result = {
                "pair_id": record["pair_id"],
                "model": model_label,
                "pid": None,
                "python_executable": sys.executable,
                "model_json": str(
                    Path(model_json).resolve()
                ),
                "model_json_sha256": sha256_file(
                    Path(model_json)
                ),
                "solver": SOLVER,
                "solver_timeout_seconds": SOLVER_TIMEOUT_SECONDS,
                "feasibility_tolerance": RETRY_FEASIBILITY_TOLERANCE,
                "condition_isolation": "fresh_model_per_condition",
                "threshold": float(threshold),
                "uptake_lower_bound": float(UPTAKE_LOWER_BOUND),
                "genes": "+".join(record["genes"]),
                "classification": "solver_error",
                "correct": False,
                "ko_status": "process_timeout",
                "ko_growth": None,
                "rescue_status": "process_timeout",
                "rescue_growth": None,
                "missing_genes": "",
                "mapped_background": "",
                "missing_background": "",
                "mapped_rescue": "",
                "missing_rescue": "",
                "associated_reactions": "",
                "bounds_after_knockout": {},
                "failure_stage": "retry_process_timeout",
            }
        else:
            retry_status = "completed"

            if completed.returncode != 0:
                raise RuntimeError(
                    f"Retry worker failed for {record['pair_id']}.\n"
                    f"stdout:\n{completed.stdout}\n"
                    f"stderr:\n{completed.stderr}"
                )

            result = parse_worker_json(
                completed.stdout,
                record["pair_id"],
            )
    else:
        if completed.returncode != 0:
            raise RuntimeError(
                f"Fresh-JSON worker failed for {record['pair_id']}.\n"
                f"stdout:\n{completed.stdout}\n"
                f"stderr:\n{completed.stderr}"
            )

        result = parse_worker_json(
            completed.stdout,
            record["pair_id"],
        )

    result.update({
        "process_timeout_seconds": PROCESS_TIMEOUT_SECONDS,
        "primary_feasibility_tolerance": PRIMARY_FEASIBILITY_TOLERANCE,
        "primary_process_timeout": primary_process_timeout,
        "retry_policy": RETRY_POLICY,
        "retry_attempted": retry_attempted,
        "retry_status": retry_status,
        "retry_feasibility_tolerance": (
            RETRY_FEASIBILITY_TOLERANCE
            if retry_attempted
            else None
        ),
        "retry_process_timeout_seconds": RETRY_PROCESS_TIMEOUT_SECONDS,
        "result_source": result_source,
        "excel_row": record["excel_row"],
        "gene_field": record["gene_field"],
        "chemical": record["chemical"],
        "n_genes": record["n_genes"],
        "strain_background": record["strain_background"],
        "reference": record["reference"],
        "conditional_medium": record["conditional_medium"],
    })

    expected_hash = sha256_file(
        Path(model_json)
    )

    if (
        result.get("model_json_sha256")
        != expected_hash
    ):
        raise RuntimeError(
            f"Worker JSON hash mismatch for {record['pair_id']}."
        )

    return result


In [4]:
# Code Cell 4 — Run a resumable fresh-process JSON benchmark
# Requires: Code Cells 1–3
# Provides: benchmark_signature and run_benchmark

def benchmark_signature(
    model_json,
    source_sbml,
    model_label,
    threshold,
    git_commit,
):
    payload = {
        "workflow_version": WORKFLOW_VERSION,
        "workflow_commit": git_commit,
        "model": model_label,
        "source_sbml_sha256": sha256_file(
            Path(source_sbml)
        ),
        "model_json_sha256": sha256_file(
            Path(model_json)
        ),
        "dataset_sha256": sha256_file(
            DATASET_XLSX
        ),
        "worker_sha256": sha256_file(
            PROCESS_WORKER
        ),
        "python": sys.version,
        "cobra": cobra.__version__,
        "solver": SOLVER,
        "solver_timeout_seconds": SOLVER_TIMEOUT_SECONDS,
        "process_timeout_seconds": PROCESS_TIMEOUT_SECONDS,
        "primary_feasibility_tolerance": PRIMARY_FEASIBILITY_TOLERANCE,
        "retry_policy": RETRY_POLICY,
        "retry_feasibility_tolerance": RETRY_FEASIBILITY_TOLERANCE,
        "retry_process_timeout_seconds": RETRY_PROCESS_TIMEOUT_SECONDS,
        "threshold": float(threshold),
        "uptake_lower_bound": UPTAKE_LOWER_BOUND,
        "mode": ISOLATION_MODE,
    }

    encoded = json.dumps(
        payload,
        sort_keys=True,
    ).encode()

    return (
        hashlib.sha256(
            encoded
        ).hexdigest(),
        payload,
    )


def run_benchmark(
    model_json,
    source_sbml,
    model_label,
    pairs_df,
    threshold,
    aliases,
    git_commit,
    resume=True,
):
    checkpoint_csv = (
        RESULTS_DIR
        / f"02_checkpoint_{model_label}_{ISOLATION_MODE}.csv"
    )
    checkpoint_json = (
        RESULTS_DIR
        / f"02_checkpoint_{model_label}_{ISOLATION_MODE}.json"
    )
    final_csv = (
        RESULTS_DIR
        / f"02_{model_label}_pair_results.csv"
    )
    final_metadata_json = (
        RESULTS_DIR
        / f"02_{model_label}_pair_results.meta.json"
    )

    signature, signature_payload = (
        benchmark_signature(
            model_json,
            source_sbml,
            model_label,
            threshold,
            git_commit,
        )
    )

    completed = pd.DataFrame()
    completed_ids = set()

    if (
        resume
        and checkpoint_csv.exists()
        and checkpoint_json.exists()
    ):
        metadata = json.loads(
            checkpoint_json.read_text(
                encoding="utf-8"
            )
        )

        if (
            metadata.get("signature")
            != signature
        ):
            raise RuntimeError(
                f"Checkpoint protocol mismatch for {model_label}."
            )

        completed = pd.read_csv(
            checkpoint_csv
        )
        completed_ids = set(
            completed[
                "pair_id"
            ].astype(str)
        )

    new_results = []
    recent_times = []
    session_start = time.perf_counter()
    total = len(pairs_df)

    if completed_ids:
        print(
            f"{model_label}: resuming from "
            f"{len(completed_ids)}/{total} completed pairs.",
            flush=True,
        )
    else:
        print(
            f"{model_label}: starting fresh-process JSON benchmark "
            f"with {total} pairs.",
            flush=True,
        )

    for record in pairs_df.to_dict(
        "records"
    ):
        if (
            record["pair_id"]
            in completed_ids
        ):
            continue

        already_done = (
            len(completed)
            + len(new_results)
        )

        print(
            f"START {already_done + 1:3d}/{total} | "
            f"{record['pair_id']} | "
            f"{record['gene_field']} | "
            f"{record['chemical']}",
            flush=True,
        )

        pair_start = time.perf_counter()

        result = run_fresh_json_pair(
            model_json,
            model_label,
            record,
            threshold,
            aliases,
        )

        pair_seconds = (
            time.perf_counter()
            - pair_start
        )

        new_results.append(
            result
        )

        current = pd.concat(
            [
                completed,
                pd.DataFrame(
                    new_results
                ),
            ],
            ignore_index=True,
        )

        current = (
            current.drop_duplicates(
                subset=["pair_id"],
                keep="last",
            )
            .sort_values(
                "excel_row"
            )
            .reset_index(
                drop=True
            )
        )

        current.to_csv(
            checkpoint_csv,
            index=False,
        )

        checkpoint_metadata = {
            "signature": signature,
            "signature_payload": signature_payload,
            "model": model_label,
            "isolation_mode": ISOLATION_MODE,
            "workflow_version": WORKFLOW_VERSION,
            "workflow_commit": git_commit,
            "completed_pairs": len(
                current
            ),
        }

        checkpoint_json.write_text(
            json.dumps(
                checkpoint_metadata,
                indent=2,
            ),
            encoding="utf-8",
        )

        recent_times.append(
            pair_seconds
        )
        recent_times = (
            recent_times[-10:]
        )

        done = len(current)
        remaining = total - done

        eta = float(
            np.mean(
                recent_times
            )
        ) * remaining

        elapsed = (
            time.perf_counter()
            - session_start
        )

        print(
            f"DONE  {done:3d}/{total} | "
            f"{record['pair_id']} | "
            f"class={result['classification']} | "
            f"KO={result['ko_status']}:{format_growth(result['ko_growth'])} | "
            f"rescue={result['rescue_status']}:{format_growth(result['rescue_growth'])} | "
            f"pid={result['pid']} | "
            f"last={format_duration(pair_seconds)} | "
            f"elapsed={format_duration(elapsed)} | "
            f"ETA={format_duration(eta)}",
            flush=True,
        )

    final = pd.concat(
        [
            completed,
            pd.DataFrame(
                new_results
            ),
        ],
        ignore_index=True,
    )

    final = (
        final.drop_duplicates(
            subset=["pair_id"],
            keep="last",
        )
        .sort_values(
            "excel_row"
        )
        .reset_index(
            drop=True
        )
    )

    if len(final) != total:
        raise AssertionError(
            f"Expected {total} completed pairs for "
            f"{model_label}, found {len(final)}."
        )

    final.to_csv(
        final_csv,
        index=False,
    )

    final_metadata = {
        "signature": signature,
        "signature_payload": signature_payload,
        "model": model_label,
        "isolation_mode": ISOLATION_MODE,
        "workflow_version": WORKFLOW_VERSION,
        "workflow_commit": git_commit,
        "completed_pairs": len(final),
        "final_csv_sha256": sha256_file(final_csv),
    }

    final_metadata_json.write_text(
        json.dumps(
            final_metadata,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        f"{model_label}: DONE ({len(final)}/{total})",
        flush=True,
    )

    return final


In [5]:
# Code Cell 5 — Build validated JSON templates and calculate threshold
# Requires: Code Cells 1–4
# Provides: validated templates, viability threshold, RUN_GIT_COMMIT

original_template = prepare_json_template(
    ORIGINAL_XML,
    ORIGINAL_JSON,
    "Yeast9",
)

curated_template = prepare_json_template(
    CURATED_XML,
    CURATED_JSON,
    "Yeast9_curated",
)

wt_growth = original_template[
    "source_wt_growth"
]

viability_threshold = (
    VIABILITY_FRACTION
    * float(wt_growth)
)

RUN_GIT_COMMIT = get_git_commit(
    ROOT
)

curated_thi6_sentinel = (
    validate_curated_thi6_sentinel(
        CURATED_JSON,
        viability_threshold,
    )
)

print("Workflow version:", WORKFLOW_VERSION)
print("Workflow commit:", RUN_GIT_COMMIT)
print("Isolation mode:", ISOLATION_MODE)
print(
    "Solver timeout:",
    SOLVER_TIMEOUT_SECONDS,
    "seconds",
)
print(
    "Process timeout:",
    PROCESS_TIMEOUT_SECONDS,
    "seconds",
)
print("WT growth:", wt_growth)
print(
    "Viability threshold:",
    viability_threshold,
)

print(
    "Original JSON WT growth:",
    original_template[
        "json_wt_growth"
    ],
)

print(
    "Curated JSON WT growth:",
    curated_template[
        "json_wt_growth"
    ],
)

print(
    "Curated THI6 sentinel:",
    curated_thi6_sentinel,
)

print(
    "Worker SHA-256:",
    sha256_file(
        PROCESS_WORKER
    ),
)


Workflow version: 0.5.8
Workflow commit: feaabea5d4408d14f15aee47da5399f5d2130be8
Isolation mode: fresh_process_json_condition_isolated
Solver timeout: 180 seconds
Process timeout: 240 seconds
WT growth: 0.08590751899388004
Viability threshold: 0.0008590751899388004
Original JSON WT growth: 0.08590751899387956
Curated JSON WT growth: 0.08583536820220818
Curated THI6 sentinel: {'gene': 'YPL214C', 'status': 'optimal', 'growth': 0.0, 'bounds': {'r_0556': [0, 0], 'r_1036': [0, 0], 'r_4753': [0, 0]}}
Worker SHA-256: 50af7194a156d83730f3d719229e4af1df540fe304edd3a8d05e98fbf69e5741


In [6]:
# Code Cell 6 — Run the 147-pair Yeast9 benchmark
# Requires: Code Cells 1–5
# Output: results/02_Yeast9_pair_results.csv and local checkpoints

original_results = run_benchmark(
    ORIGINAL_JSON,
    ORIGINAL_XML,
    "Yeast9",
    pairs,
    viability_threshold,
    ALIASES["Yeast9"],
    RUN_GIT_COMMIT,
    resume=True,
)


Yeast9: starting fresh-process JSON benchmark with 147 pairs.
START   1/147 | excel:2 | YOL058W | arginine
DONE    1/147 | excel:2 | class=correct | KO=optimal:3.75481e-18 | rescue=optimal:6.75185 | pid=22844 | last=00m 08s | elapsed=00m 08s | ETA=19m 47s
START   2/147 | excel:3 | YJL071W | arginine
DONE    2/147 | excel:3 | class=type_I | KO=optimal:0.0859075 | rescue=optimal:9.9261 | pid=19104 | last=00m 08s | elapsed=00m 16s | ETA=20m 16s
START   3/147 | excel:4 | YJL088W | arginine
DONE    3/147 | excel:4 | class=correct | KO=optimal:0 | rescue=optimal:9.9261 | pid=3796 | last=00m 07s | elapsed=00m 24s | ETA=19m 38s
START   4/147 | excel:5 | YHR018C | arginine
DONE    4/147 | excel:5 | class=correct | KO=optimal:2.67114e-18 | rescue=optimal:6.75185 | pid=25692 | last=00m 07s | elapsed=00m 31s | ETA=18m 53s
START   5/147 | excel:6 | YER069W | arginine
DONE    5/147 | excel:6 | class=correct | KO=optimal:0 | rescue=optimal:9.9261 | pid=2580 | last=00m 07s | elapsed=00m 38s | ETA=18m 

In [7]:
# Code Cell 7 — Run the 147-pair curated Yeast9 benchmark
# Requires: Code Cells 1–6
# Output: results/02_Yeast9_curated_pair_results.csv and local checkpoints

curated_results = run_benchmark(
    CURATED_JSON,
    CURATED_XML,
    "Yeast9_curated",
    pairs,
    viability_threshold,
    ALIASES["Yeast9_curated"],
    RUN_GIT_COMMIT,
    resume=True,
)


Yeast9_curated: starting fresh-process JSON benchmark with 147 pairs.
START   1/147 | excel:2 | YOL058W | arginine
DONE    1/147 | excel:2 | class=correct | KO=optimal:-1.83483e-32 | rescue=optimal:6.74085 | pid=20068 | last=00m 07s | elapsed=00m 07s | ETA=19m 17s
START   2/147 | excel:3 | YJL071W | arginine
DONE    2/147 | excel:3 | class=type_I | KO=optimal:0.0858354 | rescue=optimal:7.68234 | pid=23032 | last=00m 08s | elapsed=00m 16s | ETA=20m 19s
START   3/147 | excel:4 | YJL088W | arginine
DONE    3/147 | excel:4 | class=correct | KO=optimal:0 | rescue=optimal:7.68234 | pid=23376 | last=00m 08s | elapsed=00m 25s | ETA=19m 59s
START   4/147 | excel:5 | YHR018C | arginine
DONE    4/147 | excel:5 | class=correct | KO=optimal:-6.23289e-33 | rescue=optimal:6.74085 | pid=7828 | last=00m 07s | elapsed=00m 32s | ETA=19m 24s
START   5/147 | excel:6 | YER069W | arginine
DONE    5/147 | excel:6 | class=correct | KO=optimal:7.17411e-33 | rescue=optimal:7.68234 | pid=18872 | last=00m 07s | el

In [8]:
# Code Cell 8 — Summarize, compare, and write the run manifest
# Requires: Code Cells 1–7
# Output: benchmark summary, pairwise comparison, and 02_run_manifest.json

def validate_final_artifact(
    model_json,
    source_sbml,
    model_label,
    threshold,
    git_commit,
    expected_rows,
):
    final_csv = (
        RESULTS_DIR
        / f"02_{model_label}_pair_results.csv"
    )
    final_metadata_json = (
        RESULTS_DIR
        / f"02_{model_label}_pair_results.meta.json"
    )

    if not final_csv.exists() or not final_metadata_json.exists():
        raise RuntimeError(
            f"Missing final benchmark artifact metadata for {model_label}. "
            "Re-run the benchmark to regenerate provenance-bound outputs."
        )

    expected_signature, _ = benchmark_signature(
        model_json,
        source_sbml,
        model_label,
        threshold,
        git_commit,
    )

    metadata = json.loads(
        final_metadata_json.read_text(
            encoding="utf-8"
        )
    )

    if metadata.get("signature") != expected_signature:
        raise RuntimeError(
            f"Final benchmark protocol mismatch for {model_label}."
        )

    if metadata.get("completed_pairs") != expected_rows:
        raise RuntimeError(
            f"Final benchmark row-count metadata mismatch for {model_label}."
        )

    if (
        metadata.get("final_csv_sha256")
        != sha256_file(final_csv)
    ):
        raise RuntimeError(
            f"Final benchmark CSV hash mismatch for {model_label}."
        )


validate_final_artifact(
    ORIGINAL_JSON,
    ORIGINAL_XML,
    "Yeast9",
    viability_threshold,
    RUN_GIT_COMMIT,
    len(pairs),
)
validate_final_artifact(
    CURATED_JSON,
    CURATED_XML,
    "Yeast9_curated",
    viability_threshold,
    RUN_GIT_COMMIT,
    len(pairs),
)

def summarize(results):
    return {
        "n": len(results),
        "correct": int(
            (results["classification"] == "correct").sum()
        ),
        "type_I": int(
            (results["classification"] == "type_I").sum()
        ),
        "type_II": int(
            (results["classification"] == "type_II").sum()
        ),
        "solver_error": int(
            (results["classification"] == "solver_error").sum()
        ),
        "input_error": int(
            (results["classification"] == "input_error").sum()
        ),
    }

summary = pd.DataFrame([
    {"model": "Yeast9", **summarize(original_results)},
    {"model": "Yeast9_curated", **summarize(curated_results)},
])

summary["accuracy"] = summary["correct"] / summary["n"]

if int(summary["solver_error"].sum()) != 0:
    raise RuntimeError("Reference benchmark contains solver_error rows.")

if int(summary["input_error"].sum()) != 0:
    raise RuntimeError("Reference benchmark contains input_error rows.")

comparison = original_results.merge(
    curated_results,
    on="pair_id",
    suffixes=("_original", "_curated"),
    validate="one_to_one",
)

comparison["change"] = np.select(
    [
        (
            comparison["classification_original"] != "correct"
        )
        & (
            comparison["classification_curated"] == "correct"
        ),
        (
            comparison["classification_original"] == "correct"
        )
        & (
            comparison["classification_curated"] != "correct"
        ),
    ],
    ["fixed", "regression"],
    default="unchanged",
)

display(summary)
display(
    comparison["change"]
    .value_counts()
    .rename_axis("change")
    .to_frame("n")
)

summary.to_csv(
    RESULTS_DIR / "02_benchmark_summary.csv",
    index=False,
)

comparison.to_csv(
    RESULTS_DIR / "02_pairwise_comparison.csv",
    index=False,
)

run_manifest = {
    "workflow_version": WORKFLOW_VERSION,
    "workflow_commit": RUN_GIT_COMMIT,
    "python": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "cobra": cobra.__version__,
    "solver": SOLVER,
    "isolation_mode": ISOLATION_MODE,
    "process_timeout_seconds": PROCESS_TIMEOUT_SECONDS,
    "solver_timeout_seconds": SOLVER_TIMEOUT_SECONDS,
    "primary_feasibility_tolerance": PRIMARY_FEASIBILITY_TOLERANCE,
    "retry_policy": RETRY_POLICY,
    "retry_feasibility_tolerance": RETRY_FEASIBILITY_TOLERANCE,
    "retry_process_timeout_seconds": RETRY_PROCESS_TIMEOUT_SECONDS,
    "viability_fraction": VIABILITY_FRACTION,
    "viability_threshold": viability_threshold,
    "uptake_lower_bound": UPTAKE_LOWER_BOUND,
    "worker_sha256": sha256_file(PROCESS_WORKER),
    "original_json_sha256": sha256_file(ORIGINAL_JSON),
    "curated_json_sha256": sha256_file(CURATED_JSON),
    "curated_thi6_sentinel": curated_thi6_sentinel,
    "files": [
        {
            "file": ORIGINAL_XML.name,
            "bytes": ORIGINAL_XML.stat().st_size,
            "sha256": sha256_file(ORIGINAL_XML),
        },
        {
            "file": CURATED_XML.name,
            "bytes": CURATED_XML.stat().st_size,
            "sha256": sha256_file(CURATED_XML),
        },
        {
            "file": DATASET_XLSX.name,
            "bytes": DATASET_XLSX.stat().st_size,
            "sha256": sha256_file(DATASET_XLSX),
        },
    ],
    "summary": json.loads(summary.to_json(orient="records")),
}

(RESULTS_DIR / "02_run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2),
    encoding="utf-8",
)


            model    n  correct  type_I  type_II  solver_error  input_error  accuracy
0          Yeast9  147       92      37       18             0            0  0.625850
1  Yeast9_curated  147      116      28        3             0            0  0.789116
              n
change         
unchanged   119
fixed        26
regression    2


## Interpretation checkpoint

Do not interpret the benchmark if `solver_error` or `input_error` is non-zero. The paper reports 93/147 correct predictions for Yeast9 and 117/147 for the curated model; these are comparison targets, not forced expected outputs.

Workflow 0.5.5 uses a fresh Python process and a fresh JSON-reconstructed COBRA model for every phenotype. The released SBML is parsed only once per model to create the JSON template. The curated THI6/YPL214C knockout is used as a sentinel before the batch run because previous `model.copy()` and sequential-context experiments produced solver/model inconsistencies for this phenotype.
